## Libraries

In [1]:
import numpy as np
import pandas as pd

from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# NEW: cuML GPU Random Forest
from cuml.ensemble import RandomForestRegressor as cuRFRegressor

## Config

In [2]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")   # all dates >= this are test

# <<< FILL THESE FROM CV RESULTS >>>

best_lag_set = [1, 12, 24]   
best_params  = {
    "n_estimators": 250,
    "max_depth": 10,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
}

# continuous features
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

# one-hot / categorical features
categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

## Metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(residuals, xs, ys, k=5, eps=1e-8):
    """
    Simple Moran's I using k-nearest neighbours with inverse-distance weights.
    residuals: shape (N,)
    xs, ys: coordinates aligned with residuals.
    """
    residuals = np.asarray(residuals)
    N = len(residuals)
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N))
    for i in range(N):
        neigh_idx = indices[i, 1:]  # skip self
        w = 1.0 / (distances[i, 1:] + eps)
        W[i, neigh_idx] = w

    S0 = W.sum()
    num = 0.0
    for i in range(N):
        for j in range(N):
            num += W[i, j] * x_dev[i] * x_dev[j]
    den = np.sum(x_dev ** 2) + eps
    I = (N / S0) * (num / den)
    return I

## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])


## Evaluation

In [5]:

# =========================================================
# LEAK-FREE STL: fit on TRAIN ONLY, extend into TEST
# =========================================================
df["stl_trend"] = np.nan
df["stl_seasonal"] = np.nan
df["stl_resid"] = np.nan

for la, sub in df.groupby(ENTITY_COL):
    sub = sub.sort_values(TIME_COL)

    # training portion for STL (can include pre-2007 if present, as long as <= TRAIN_END_DATE)
    train_sub = sub[sub[TIME_COL] <= TRAIN_END_DATE]
    if len(train_sub) < 24:
        continue

    series = train_sub[TARGET_COL].astype(float)
    stl = STL(series, period=12, robust=True)
    res = stl.fit()

    # assign training STL components
    df.loc[train_sub.index, "stl_trend"] = res.trend
    df.loc[train_sub.index, "stl_seasonal"] = res.seasonal
    df.loc[train_sub.index, "stl_resid"] = res.resid

    # extend into future (test period and any post-train dates if present)
    future_sub = sub[sub[TIME_COL] > TRAIN_END_DATE].copy()
    if future_sub.empty:
        continue

    n_future = len(future_sub)

    # seasonal: repeat last 12-month pattern
    season_train = res.seasonal
    if len(season_train) >= 12:
        base_pattern = season_train[-12:]
    else:
        base_pattern = season_train
    reps = int(np.ceil(n_future / len(base_pattern)))
    season_future = np.tile(base_pattern, reps)[:n_future]

    # trend: linear extrapolation
    trend_train = res.trend
    t_idx = np.arange(len(trend_train))
    if len(trend_train) >= 2:
        coef = np.polyfit(t_idx, trend_train, 1)
        future_t = np.arange(len(trend_train), len(trend_train) + n_future)
        trend_future = coef[0] * future_t + coef[1]
    else:
        trend_future = np.full(n_future, trend_train[-1])

    df.loc[future_sub.index, "stl_trend"] = trend_future
    df.loc[future_sub.index, "stl_seasonal"] = season_future
    # residuals unknown in future – keep as 0
    df.loc[future_sub.index, "stl_resid"] = 0.0

# =========================================================
# LAGGED STL FEATURES (best_lag_set) ACROSS FULL PANEL
# =========================================================
required_lag_cols = []
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        col = f"{comp}_lag{lag}"
        df[col] = df.groupby(ENTITY_COL)[comp].shift(lag)
        required_lag_cols.append(col)

# =========================================================
# TRAIN / TEST SPLIT (feature period starts April 2007)
# =========================================================
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_test  = (df[TIME_COL] >= TEST_START_DATE)

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# Need all lag columns present
df_train = df_train.dropna(subset=required_lag_cols)
df_test  = df_test.dropna(subset=required_lag_cols)

# =========================================================
# STANDARDISE CONTINUOUS + LAG FEATURES
# =========================================================
scale_cols = continuous_cols + required_lag_cols
scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
df_test[scale_cols]  = scaler.transform(df_test[scale_cols])

# =========================================================
# BUILD MATRICES
# =========================================================
feature_cols = continuous_cols + categorical_cols + required_lag_cols

X_train = df_train[feature_cols].to_numpy(dtype=np.float32)
y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32)

X_test  = df_test[feature_cols].to_numpy(dtype=np.float32)
y_test  = df_test[TARGET_COL].to_numpy(dtype=np.float32)


# =========================================================
# TRAIN FINAL RANDOM FOREST
# =========================================================
cuml_params = best_params.copy()
# if you ever want "no max depth" from CV, set max_depth = -1 for cuML

rf = cuRFRegressor(
    **cuml_params,
    random_state=42,
    n_streams=8,        # parallel CUDA streams
    output_type="numpy" # return numpy arrays
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)  # numpy array

df_test["y_pred"] = y_pred

# =========================================================
# GLOBAL ACCURACY
# =========================================================
global_mae   = mae(y_test, y_pred)
global_rmse  = rmse(y_test, y_pred)
global_smape = smape(y_test, y_pred)
global_mase  = mase(y_test, y_pred, y_train, m=12)

print("=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test
    .groupby(ENTITY_COL)
    .apply(lambda g: mae(g[TARGET_COL], g["y_pred"]))
)

median_mae = la_mae.median()
p75_mae    = la_mae.quantile(0.75)

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I: mean residual per LA over test period
df_test["resid"] = df_test[TARGET_COL] - df_test["y_pred"]

# Mean residual per LA over test period
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

# --- Robust centroid extraction (avoids pre-2007 NaNs) ---
centroids = (
    df_test
        .dropna(subset=["centroid_x", "centroid_y"])   # remove rows with missing centroids
        .sort_values(TIME_COL)
        .groupby(ENTITY_COL)
        .tail(1)                                       # take latest valid centroid per LA
        .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
        .loc[la_resid_mean.index]
)

# --- SAFETY FILTER (this is the mask you were missing) ---
mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

# Optional sanity check
print(f"LAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

# --- Compute Moran's I safely ---
N_valid = len(centroids_valid)
if N_valid <= 1:
    I_moran = np.nan
else:
    k_effective = min(5, N_valid - 1)

    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_effective
    )

print("\n=== Spatio-temporal diagnostics ===")
print(f"Moran's I (mean residuals across LAs): {I_moran:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = (
    df_test
    .groupby(TIME_COL)["resid"]
    .mean()
    .sort_index()
)

lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)

# Extract the first (and only) row as floats
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])

print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# OPTIONAL: DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


=== Global accuracy ===
MAE   : 18,049.916
RMSE  : 31,594.010
sMAPE : 5.283%
MASE  : 1.479

=== Across-LA consistency ===
Median LA MAE       : 13,730.756
75th percentile MAE : 17,873.915
LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): 0.6481
Ljung–Box Q(12): stat=143.086, p=0.0000

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.255
Growth-rate error MAE (12-month)  : 0.0348


/tmp/ipykernel_31571/2075073600.py:137: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mae(g[TARGET_COL], g["y_pred"]))


## Result output

In [7]:

output_path = "../../results/rf_gpu_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "RandomForest",
    "lag_set": str(best_lag_set),
    "params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": I_moran,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")



Results saved to: ../../results/rf_gpu_final_test_results.xlsx
